# 04 — Structured Outputs and Typed Interfaces

Northstar converts a request into a typed case brief. This offline lab demonstrates why valid JSON is not enough.

## Objective and boundary

Validate syntax, schema shape, and business semantics outside the model. No API or side effect is used.

In [1]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys
path=Path.cwd()/'curriculum/beginner/04-structured-outputs-and-typed-interfaces/lab.py'
if not path.exists(): path=Path.cwd()/'lab.py'
spec=spec_from_file_location('typed_lab',path); lab=module_from_spec(spec); sys.modules[spec.name]=lab; spec.loader.exec_module(lab)

## Baseline and experiment

Prompt-only JSON has no application-controlled guarantee. Run the same validator over malformed, schema-invalid, and semantically unsupported candidates.

In [2]:
results={name:lab.validate(raw) for name,raw in lab.SAMPLES.items()}
results

{'valid': (True, 'valid'),
 'malformed': (False, 'invalid JSON'),
 'enum': (False, 'enum violation'),
 'extra': (False, 'schema fields do not match'),
 'semantic': (False, 'semantic/evidence violation')}

## Failure injection

The semantic response has valid JSON and every required field, but makes an unsupported eligibility claim. Semantic validation must reject it.

In [3]:
assert lab.validate(lab.SAMPLES['semantic']) == (False,'semantic/evidence violation')

## Bounded repair and production

Repair an extra-field response once. Malformed JSON and invalid enums require clarification, escalation, or one separately evaluated retry; never loop indefinitely. Track parse, schema, semantic, repair, and escalation rates separately.

Exercises: add a nested object, a missing field, and a backwards-compatible schema migration. Valid data shape never proves an authorized or supported business decision.

In [4]:
repaired=lab.bounded_repair(lab.SAMPLES['extra'])
assert repaired is not None
lab.validate(repaired), repaired

((True, 'valid'),
 '{"intent": "refund", "answer": "Review it.", "evidence_id": "refund-v3", "needs_human": false}')